# Adding the Dependecy

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.7 MB/s eta 0:00:00


# Importing Modules

In [ ]:
import os
import cv2
import kagglehub
from datetime import datetime
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Downloading Image Dataset

In [ ]:
# --- 1. Download Dataset ---
path = kagglehub.dataset_download("kapillondhe/american-sign-language")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'american-sign-language' dataset.
Path to dataset files: /kaggle/input/american-sign-language


# Fetching 5000 Images

In [ ]:
import os
import shutil
import random

# Source and destination
source_train = '/kaggle/input/american-sign-language/ASL_Dataset/Train'
dest_root = '/content/asl_data_5000'
limit_per_class = 179

# Function to copy limited files from Train
def copy_train_data(source_dir, dest_dir, limit):
    # Ensure destination root exists
    os.makedirs(dest_dir, exist_ok=True)

    # Loop through each class folder (A, B, C...)
    for class_name in os.listdir(source_dir):
        class_source_path = os.path.join(source_dir, class_name)
        class_dest_path = os.path.join(dest_dir, class_name)

        # Only process if it is a directory
        if os.path.isdir(class_source_path):
            os.makedirs(class_dest_path, exist_ok=True)

            # Get list of all images
            images = [f for f in os.listdir(class_source_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

            # Select 250 or the max available if less than 250
            count_to_copy = min(len(images), limit)
            selected_images = random.sample(images, count_to_copy)

            for img in selected_images:
                shutil.copy2(os.path.join(class_source_path, img), os.path.join(class_dest_path, img))

            print(f"Copied {count_to_copy} images for class: {class_name}")

# Execute the process
print("Processing Train folder only...")
copy_train_data(source_train, dest_root, limit_per_class)

print(f"\n✅ Dataset preparation complete. Files saved to {dest_root}")

Processing Train folder only...
Copied 179 images for class: N
Copied 179 images for class: R
Copied 179 images for class: Space
Copied 179 images for class: B
Copied 179 images for class: I
Copied 179 images for class: F
Copied 179 images for class: H
Copied 179 images for class: E
Copied 179 images for class: U
Copied 179 images for class: M
Copied 179 images for class: X
Copied 179 images for class: K
Copied 179 images for class: Q
Copied 179 images for class: Y
Copied 179 images for class: S
Copied 179 images for class: G
Copied 179 images for class: A
Copied 179 images for class: O
Copied 179 images for class: T
Copied 179 images for class: V
Copied 179 images for class: Z
Copied 179 images for class: C
Copied 179 images for class: P
Copied 179 images for class: L
Copied 179 images for class: W
Copied 179 images for class: D
Copied 179 images for class: Nothing
Copied 179 images for class: J

✅ Dataset preparation complete. Files saved to /content/asl_data_5000


In [ ]:
import os

target_dir = '/content/asl_data_5000'
valid_extensions = ('.jpg', '.jpeg', '.png')
total_images = 0

print(f"{'Class Name':<15} | {'Image Count'}")
print("-" * 30)

# Iterate through class folders
for class_name in sorted(os.listdir(target_dir)):
    class_path = os.path.join(target_dir, class_name)

    if os.path.isdir(class_path):
        # Count files in the current class folder
        count = len([f for f in os.listdir(class_path) if f.lower().endswith(valid_extensions)])
        total_images += count
        print(f"{class_name:<15} | {count}")

print("-" * 30)
print(f"Total images found: {total_images}")

Class Name      | Image Count
------------------------------
A               | 179
B               | 179
C               | 179
D               | 179
E               | 179
F               | 179
G               | 179
H               | 179
I               | 179
J               | 179
K               | 179
L               | 179
M               | 179
N               | 179
Nothing         | 179
O               | 179
P               | 179
Q               | 179
R               | 179
S               | 179
Space           | 179
T               | 179
U               | 179
V               | 179
W               | 179
X               | 179
Y               | 179
Z               | 179
------------------------------
Total images found: 5012


# Spliting Train & Val

In [ ]:
import os
import shutil
import random

source_data = '/content/asl_data_5000'
train_dir = '/content/master_asl_data/train'
val_dir = '/content/master_asl_data/val'
split_ratio = 0.8

# Explicitly define classes or filter out existing train/val directories
for class_name in os.listdir(source_data):
    # SKIP if the folder is named 'train' or 'val'
    if class_name.lower() in ['train', 'val']:
        continue

    class_source_path = os.path.join(source_data, class_name)

    if os.path.isdir(class_source_path):
        # Create corresponding folders
        os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
        os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)

        # Get all images
        images = [f for f in os.listdir(class_source_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

        if not images:
            print(f"Skipping {class_name}: No images found.")
            continue

        random.shuffle(images)

        # Determine split point
        split_idx = int(len(images) * split_ratio)
        train_imgs = images[:split_idx]
        val_imgs = images[split_idx:]

        # Copy files
        for img in train_imgs:
            shutil.copy2(os.path.join(class_source_path, img), os.path.join(train_dir, class_name, img))
        for img in val_imgs:
            shutil.copy2(os.path.join(class_source_path, img), os.path.join(val_dir, class_name, img))

        print(f"Split {class_name}: {len(train_imgs)} train, {len(val_imgs)} val")

print("\n✅ Splitting complete! Structure is ready for training.")

Split Q: 143 train, 36 val
Split V: 143 train, 36 val
Split T: 143 train, 36 val
Split I: 143 train, 36 val
Split L: 143 train, 36 val
Split W: 143 train, 36 val
Split Space: 143 train, 36 val
Split R: 143 train, 36 val
Split S: 143 train, 36 val
Split A: 143 train, 36 val
Split H: 143 train, 36 val
Split E: 143 train, 36 val
Split X: 143 train, 36 val
Split D: 143 train, 36 val
Split N: 143 train, 36 val
Split O: 143 train, 36 val
Split B: 143 train, 36 val
Split K: 143 train, 36 val
Split U: 143 train, 36 val
Split Y: 143 train, 36 val
Split Z: 143 train, 36 val
Split J: 143 train, 36 val
Split Nothing: 143 train, 36 val
Split M: 143 train, 36 val
Split P: 143 train, 36 val
Split F: 143 train, 36 val
Split C: 143 train, 36 val
Split G: 143 train, 36 val

✅ Splitting complete! Structure is ready for training.


# Data_image.yaml

In [ ]:
yaml_content = """
path: /content/asl_data_5000
train: train
val: val
names:
  0: A
  1: B
  2: C
  3: D
  4: E
  5: F
  6: G
  7: H
  8: I
  9: J
  10: K
  11: L
  12: M
  13: N
  14: Nothing
  15: O
  16: P
  17: Q
  18: R
  19: S
  20: Space
  21: T
  22: U
  23: V
  24: W
  25: X
  26: Y
  27: Z
"""

with open('/content/data_image.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data_image.yaml created successfully.")

✅ data.yaml created successfully.


# Transfer Learning - Training Model

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

# 1. Initialize
model = YOLO('yolov8n-cls.pt')

# 2. Train
results = model.train(
    data='/content/master_asl_data',
    epochs=1,
    imgsz=224,
    batch=1,
    project='asl_training',
    name='asl_model_V1'
)

# 3. Dynamic path finding (Fail-safe approach)
# Search globally within 'runs/' to handle any environment-specific pathing
run_dirs = glob.glob('/content/runs/classify/asl_training/asl_model_V1*')

if not run_dirs:
    # Fallback search if the above path doesn't exist
    run_dirs = glob.glob('/content/runs/classify/**/*')

if run_dirs:
    latest_run = max(run_dirs, key=os.path.getmtime)
    print(f"✅ Automatically detected latest run: {latest_run}")
else:
    raise FileNotFoundError("❌ Could not find any training results in the 'runs/' directory.")

# 4. Plot Training/Validation Loss
csv_path = os.path.join(latest_run, 'results.csv')

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    plt.figure(figsize=(10, 5))
    plt.plot(df['epoch'], df['train/loss'], label='Train Loss')
    plt.plot(df['epoch'], df['val/loss'], label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 5. Confusion Matrix
    cm_path = os.path.join(latest_run, 'confusion_matrix.png')
    if os.path.exists(cm_path):
        plt.figure(figsize=(8, 8))
        plt.imshow(plt.imread(cm_path))
        plt.axis('off')
        plt.title('Confusion Matrix')
        plt.show()
    else:
        print("ℹ️ Confusion matrix not generated. This often happens if validation is skipped.")
else:
    print(f"❌ Could not find results.csv in {latest_run}")

# 6. Save final model
best_model_path = os.path.join(latest_run, 'weights', 'best.pt')
destination_path = '/content/final_asl_model.pt'

if os.path.exists(best_model_path):
    shutil.copy(best_model_path, destination_path)
    print(f"✅ Model successfully saved to {destination_path}")
else:
    print(f"❌ Error: Could not find weights at {best_model_path}")

In [ ]:
from google.colab import files

# Define the path to your saved model
model_path = '/content/final_asl_model.pt'

# Trigger the download
files.download(model_path)

# Fetching Test Data

In [ ]:
import os
import shutil

# Source and destination for Test data
source_test = '/kaggle/input/american-sign-language/ASL_Dataset/Test'
dest_test_root = '/content/asl_test_data'

def copy_all_test_data(source_dir, dest_dir):
    # Ensure destination root exists
    os.makedirs(dest_dir, exist_ok=True)

    # Loop through each class folder (A, B, C...)
    for class_name in os.listdir(source_dir):
        class_source_path = os.path.join(source_dir, class_name)
        class_dest_path = os.path.join(dest_dir, class_name)

        # Only process if it is a directory
        if os.path.isdir(class_source_path):
            os.makedirs(class_dest_path, exist_ok=True)

            # Get list of all images
            images = [f for f in os.listdir(class_source_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

            # Copy all found images
            for img in images:
                shutil.copy2(os.path.join(class_source_path, img), os.path.join(class_dest_path, img))

            print(f"Copied {len(images)} images for Test class: {class_name}")

# Execute the process
print("Processing ALL images from Test folder...")
copy_all_test_data(source_test, dest_test_root)

print(f"\n✅ All test data prepared. Files saved to {dest_test_root}")

Processing ALL images from Test folder...
Copied 4 images for Test class: N
Copied 4 images for Test class: R
Copied 4 images for Test class: Space
Copied 4 images for Test class: B
Copied 4 images for Test class: I
Copied 4 images for Test class: F
Copied 4 images for Test class: H
Copied 4 images for Test class: E
Copied 4 images for Test class: U
Copied 4 images for Test class: M
Copied 4 images for Test class: X
Copied 4 images for Test class: K
Copied 4 images for Test class: Q
Copied 4 images for Test class: Y
Copied 4 images for Test class: S
Copied 4 images for Test class: G
Copied 4 images for Test class: A
Copied 4 images for Test class: O
Copied 4 images for Test class: T
Copied 4 images for Test class: V
Copied 4 images for Test class: Z
Copied 4 images for Test class: C
Copied 4 images for Test class: P
Copied 4 images for Test class: L
Copied 4 images for Test class: W
Copied 4 images for Test class: D
Copied 4 images for Test class: Nothing
Copied 4 images for Test class

# Testing Model

In [ ]:
from ultralytics import YOLO
import os
import shutil

# 1. Load your model
model = YOLO('final_asl_model.pt')

# 2. Paths
test_data_path = '/content/asl_test_data'
results_folder = '/content/results'

print(f"🔍 Scanning {test_data_path} for subfolders...")

# 3. Process subfolders
# 'class_name' will be the label (A, B, C...)
for class_name in os.listdir(test_data_path):
    class_path = os.path.join(test_data_path, class_name)

    # Only process directories
    if os.path.isdir(class_path):
        print(f"Testing images from ground-truth class: {class_name}")

        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
                img_path = os.path.join(class_path, img_name)

                # Predict
                results = model.predict(source=img_path, verbose=False)
                pred_class = results[0].names[results[0].probs.top1]

                # Define destination: pred_{model_prediction}
                # We store it inside a folder named after what the model thought it was
                target_folder = os.path.join(results_folder, f"pred_{pred_class}")
                os.makedirs(target_folder, exist_ok=True)

                # Save with prefix to remember the true label
                # Filename format: actual_{TrueLabel}_{OriginalName}
                new_name = f"actual_{class_name}_{img_name}"
                shutil.copy2(img_path, os.path.join(target_folder, new_name))

print(f"\n🎉 Testing complete! Predictions are sorted in {results_folder}")

🔍 Scanning /content/asl_test_data for subfolders...
Testing images from ground-truth class: Q
Testing images from ground-truth class: V
Testing images from ground-truth class: T
Testing images from ground-truth class: I
Testing images from ground-truth class: L
Testing images from ground-truth class: W
Testing images from ground-truth class: Space
Testing images from ground-truth class: R
Testing images from ground-truth class: S
Testing images from ground-truth class: A
Testing images from ground-truth class: H
Testing images from ground-truth class: E
Testing images from ground-truth class: X
Testing images from ground-truth class: D
Testing images from ground-truth class: N
Testing images from ground-truth class: O
Testing images from ground-truth class: B
Testing images from ground-truth class: K
Testing images from ground-truth class: U
Testing images from ground-truth class: Y
Testing images from ground-truth class: Z
Testing images from ground-truth class: J
Testing images from 

# Visualization

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# 1. Paths
results_root = '/content/results'

actuals = []
predictions = []

# 2. Parse folders to collect labels
# Folder name is 'pred_X', filename prefix is 'actual_Y_'
for pred_folder in os.listdir(results_root):
    if pred_folder.startswith('pred_'):
        pred_label = pred_folder.replace('pred_', '')

        folder_path = os.path.join(results_root, pred_folder)
        for img_name in os.listdir(folder_path):
            if img_name.startswith('actual_'):
                # Extract actual label from filename: actual_{label}_{name}
                actual_label = img_name.split('_')[1]

                actuals.append(actual_label)
                predictions.append(pred_label)

# 3. Generate Classification Report
print("--- Classification Report ---")
print(classification_report(actuals, predictions))

# 4. Generate Confusion Matrix
cm = confusion_matrix(actuals, predictions)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(list(set(predictions))),
            yticklabels=sorted(list(set(actuals))))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: ASL Model Performance')
plt.show()

# Custom Model - From Scrartch

# Model Definition

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ASLModel(nn.Module):
    def __init__(self, num_classes):
        super(ASLModel, self).__init__()
        # 6-Layer Architecture: 4 Conv Layers, 2 Fully Connected Layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 512) # Assuming input images are 64x64
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))
        x = x.view(-1, 128 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
import shutil
import os

# Path to the offending folder
checkpoint_path = '/content/master_asl_data/.ipynb_checkpoints'

if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
    print("✅ Removed .ipynb_checkpoints folder.")
else:
    print("ℹ️ .ipynb_checkpoints not found, proceeding.")

✅ Removed .ipynb_checkpoints folder.


# Loading Data

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load datasets
train_dataset = datasets.ImageFolder(root='/content/master_asl_data/train', transform=transform)
val_dataset = datasets.ImageFolder(root='/content/master_asl_data/val', transform=transform)

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"✅ Loaded {len(train_dataset)} training images and {len(val_dataset)} validation images.")

✅ Loaded 4004 training images and 1008 validation images.


# Model Training

In [ ]:
import torch
import torch.optim as optim

# Initialize lists to store metrics
train_losses = []
val_losses = []
val_accuracies = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Define Model: The neural network architecture
# Use the length of the dataset classes directly
num_classes = len(train_dataset.classes)
print(f"Initializing model for {num_classes} classes.")

# Initialize the model with the correct count
model = ASLModel(num_classes=num_classes).to(device)

# 2. Define Criterion: The loss function (CrossEntropy is standard for classification)
criterion = nn.CrossEntropyLoss()

# 3. Define Optimizer: The algorithm that updates model weights (Adam is recommended)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Assuming model, criterion, optimizer, train_loader, and val_loader are defined
print("🚀 Starting training with loss tracking...")

for epoch in range(20): # Set to your desired number of epochs
    # --- Training Phase ---
    model.train()
    running_train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    # Calculate average training loss for the epoch
    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # --- Validation Phase ---
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_acc = 100 * correct / total

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_acc)

    print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_acc:.2f}%")

# Save the final model
torch.save(model.state_dict(), '/content/final_asl_model.pth')
print("\n✅ Training complete and model saved.")

# Visualization

In [ ]:
import matplotlib.pyplot as plt

# 1. Define the function first
def plot_metrics(train_losses, val_losses, val_accuracies):
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(train_losses, label='Train Loss', color='blue')
    ax1.plot(val_losses, label='Val Loss', color='orange')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend(loc='upper left')

    ax2 = ax1.twinx()
    ax2.plot(val_accuracies, label='Val Accuracy', color='green', linestyle='--')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend(loc='upper right')

    plt.title('Training Performance Metrics')
    plt.show()

# 2. RUN TRAINING LOOP HERE...
# (The code I provided in the previous response populates
# train_losses, val_losses, and val_accuracies lists inside the loop)

# 3. CALL THE FUNCTION AFTER THE LOOP FINISHES
plot_metrics(train_losses, val_losses, val_accuracies)

# Testing Data

In [ ]:
import torch
import os
import shutil
from torchvision import transforms
from PIL import Image

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Define Model Architecture (Ensure this matches your training structure exactly)
# Make sure you have the 'ASLModel' class definition in this cell as well
model = ASLModel(num_classes=num_classes)
model.load_state_dict(torch.load('/content/final_asl_model.pth', map_location=device))
model.to(device)
model.eval()

# 3. Define Transforms (Must match training normalization)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 4. Map class indices to names
# This assumes your train folders are sorted A-Z
class_names = sorted(os.listdir('/content/master_asl_data/train'))
idx_to_class = {i: name for i, name in enumerate(class_names)}

# 5. Testing logic
test_data_path = '/content/asl_test_data'
results_folder = '/content/results_custom'

print(f"🚀 Starting inference on {test_data_path}...")

for class_label in os.listdir(test_data_path):
    class_path = os.path.join(test_data_path, class_label)

    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
                img_path = os.path.join(class_path, img_name)

                # Inference
                img = Image.open(img_path).convert('RGB')
                input_tensor = transform(img).unsqueeze(0).to(device)

                with torch.no_grad():
                    outputs = model(input_tensor)
                    _, predicted = torch.max(outputs, 1)
                    pred_label = idx_to_class[predicted.item()]

                # Sort into results folder
                target_folder = os.path.join(results_folder, f"pred_{pred_label}")
                os.makedirs(target_folder, exist_ok=True)
                shutil.copy2(img_path, os.path.join(target_folder, f"actual_{class_label}_{img_name}"))

print(f"\n🎉 Testing complete! Results sorted into: {results_folder}")

🚀 Starting inference on /content/asl_test_data...

🎉 Testing complete! Results sorted into: /content/results_custom


# Visualization

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Update this path if your results are in a different location (e.g., '/content/results')
results_root = '/content/results_custom'

if not os.path.exists(results_root):
    print(f"Directory {results_root} not found. Please ensure your inference script successfully created this folder.")
else:
    actuals = []
    predictions = []

    # Parse folders: folder is 'pred_X', file is 'actual_Y_...'
    for pred_folder in os.listdir(results_root):
        if pred_folder.startswith('pred_'):
            pred_label = pred_folder.replace('pred_', '')
            folder_path = os.path.join(results_root, pred_folder)

            for img_name in os.listdir(folder_path):
                if img_name.startswith('actual_'):
                    parts = img_name.split('_')
                    if len(parts) >= 2:
                        actual_label = parts[1]
                        actuals.append(actual_label)
                        predictions.append(pred_label)

    if len(actuals) > 0:
        # Print Classification Report
        print("--- Classification Report ---")
        print(classification_report(actuals, predictions))

        # Generate Confusion Matrix
        labels = sorted(list(set(actuals) | set(predictions)))
        cm = confusion_matrix(actuals, predictions, labels=labels)

        # Plot Confusion Matrix
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=labels, yticklabels=labels)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title('Confusion Matrix: Custom ASL Model')
        plt.tight_layout()
        plt.savefig('confusion_matrix.png')
        print("✅ Visualization saved as confusion_matrix.png")
    else:
        print("No valid files found to generate metrics.")

# Video Analysis

# Importing Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("asthalochanmohanta/american-sign-language-asl")

print("Path to dataset files:", path)

100%|██████████| 3.11G/3.11G [00:55<00:00, 60.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/asthalochanmohanta/american-sign-language-asl/versions/1


In [ ]:
import shutil
import os

# Create a visible folder
os.makedirs('/content/asl_data_video', exist_ok=True)

# Copy files from the hidden cache to the visible folder
# 'path' is the variable from your previous cell
shutil.copytree(path, '/content/asl_data_video', dirs_exist_ok=True)

print("Data successfully moved to /content/asl_data_video")
!ls /content/asl_data_video

Data successfully moved to /content/asl_data_video
dataset_v2


# Video Split to Frames

In [ ]:
import cv2
import os

# Use the path from your kagglehub download
# source = path
# output = '/content/asl_video_ready'

def prepare_yolo_dataset(source_root, output_root):
    # 1. Check if source exists
    if not os.path.exists(source_root):
        print(f"ERROR: Source path does not exist: {source_root}")
        return

    for split in ['train', 'val', 'test']:
        split_src = os.path.join(source_root, split)

        if not os.path.exists(split_src):
            print(f"Skipping {split}: Folder not found at {split_src}")
            continue

        print(f"Processing {split} split...")

        for word_folder in os.listdir(split_src):
            word_path = os.path.join(split_src, word_folder)
            if not os.path.isdir(word_path): continue

            dest_path = os.path.join(output_root, split, word_folder)
            os.makedirs(dest_path, exist_ok=True)

            videos = [f for f in os.listdir(word_path) if f.lower().endswith(('.mp4', '.avi', '.mov'))]
            print(f"  Found {len(videos)} videos for word: {word_folder}")

            for vid_name in videos:
                vid_path = os.path.join(word_path, vid_name)
                cap = cv2.VideoCapture(vid_path)

                f_idx = 0
                while True:
                    success, frame = cap.read()
                    if not success: break

                    if f_idx % 5 == 0:
                        img_name = f"{vid_name.split('.')[0]}_f{f_idx}.jpg"
                        cv2.imwrite(os.path.join(dest_path, img_name), frame)
                    f_idx += 1
                cap.release()
    print("Done! Check the folder icon on the left of Colab.")

# EXECUTING:
# Ensure 'path' is the variable from your kagglehub.dataset_download
prepare_yolo_dataset('/content/asl_data_video/dataset_v2', '/content/asl_video_ready')

Processing train split...
  Found 60 videos for word: help
  Found 46 videos for word: want
  Found 61 videos for word: book
  Found 79 videos for word: meet
  Found 60 videos for word: happy
  Found 59 videos for word: forget
  Found 54 videos for word: yes
  Found 76 videos for word: like
  Found 70 videos for word: what
  Found 58 videos for word: go
  Found 72 videos for word: bathroom
  Found 60 videos for word: fine
  Found 77 videos for word: sad
  Found 46 videos for word: wrong
  Found 54 videos for word: father
  Found 64 videos for word: my
  Found 60 videos for word: finish
  Found 60 videos for word: more
  Found 59 videos for word: right
  Found 52 videos for word: please
  Found 56 videos for word: good
  Found 56 videos for word: hello
  Found 52 videos for word: no
  Found 66 videos for word: why
  Found 61 videos for word: where
  Found 64 videos for word: thank you
  Found 60 videos for word: which
  Found 91 videos for word: name
  Found 62 videos for word: nice
  F

# Transfer Learning and Training Model

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

# 1. Initialize (using a pre-trained classification model)
model = YOLO('yolov8n-cls.pt')

# 2. Train on your new video-ready dataset
# Ensure /content/asl_video_ready contains subfolders (train/val)
# or a data.yaml file if required by your setup.
results = model.train(
    data='/content/asl_video_ready',
    epochs=20,           # Increased from 1 for better results
    imgsz=224,
    batch=16,            # Increased batch size for faster, more stable training
    project='asl_training',
    name='asl_model_video'
)

# 3. Dynamic path finding
# YOLOv8 usually saves to '/content/runs/classify/...'
run_dirs = glob.glob('/content/runs/classify/asl_training/asl_model_video*')
latest_run = max(run_dirs, key=os.path.getmtime) if run_dirs else None

if latest_run:
    print(f"✅ Automatically detected latest run: {latest_run}")
else:
    raise FileNotFoundError("❌ Could not find training results.")

# 4. Plot Training/Validation Loss
csv_path = os.path.join(latest_run, 'results.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    plt.figure(figsize=(10, 5))
    plt.plot(df['epoch'], df['metrics/loss'] if 'metrics/loss' in df else df['train/loss'], label='Train Loss')
    plt.plot(df['epoch'], df['val/loss'], label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

# 5. Save final model
best_model_path = os.path.join(latest_run, 'weights', 'best.pt')
destination_path = '/content/final_asl_video_model.pt'

if os.path.exists(best_model_path):
    shutil.copy(best_model_path, destination_path)
    print(f"✅ Model successfully saved to {destination_path}")

In [ ]:
from google.colab import files

# Define the path to your saved model
model_path = '/content/final_asl_model.pt'

# Trigger the download
files.download(model_path)

# Model Testing

In [ ]:
from ultralytics import YOLO
import os
import shutil

# 1. Load your model
model = YOLO('final_asl_video_model.pt')

# 2. Paths
test_data_path = '/content/asl_video_ready/test'
results_folder = '/content/results_video'

print(f"🔍 Scanning {test_data_path} for subfolders...")

# 3. Process subfolders
# 'class_name' will be the label (A, B, C...)
for class_name in os.listdir(test_data_path):
    class_path = os.path.join(test_data_path, class_name)

    # Only process directories
    if os.path.isdir(class_path):
        print(f"Testing images from ground-truth class: {class_name}")

        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
                img_path = os.path.join(class_path, img_name)

                # Predict
                results = model.predict(source=img_path, verbose=False)
                pred_class = results[0].names[results[0].probs.top1]

                # Define destination: pred_{model_prediction}
                # We store it inside a folder named after what the model thought it was
                target_folder = os.path.join(results_folder, f"pred_{pred_class}")
                os.makedirs(target_folder, exist_ok=True)

                # Save with prefix to remember the true label
                # Filename format: actual_{TrueLabel}_{OriginalName}
                new_name = f"actual_{class_name}_{img_name}"
                shutil.copy2(img_path, os.path.join(target_folder, new_name))

print(f"\n🎉 Testing complete! Predictions are sorted in {results_folder}")

# Visualization

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# 1. Paths
results_root = '/content/results_video'

actuals = []
predictions = []

# 2. Parse folders to collect labels
# Folder name is 'pred_X', filename prefix is 'actual_Y_'
for pred_folder in os.listdir(results_root):
    if pred_folder.startswith('pred_'):
        pred_label = pred_folder.replace('pred_', '')

        folder_path = os.path.join(results_root, pred_folder)
        for img_name in os.listdir(folder_path):
            if img_name.startswith('actual_'):
                # Extract actual label from filename: actual_{label}_{name}
                actual_label = img_name.split('_')[1]

                actuals.append(actual_label)
                predictions.append(pred_label)

# 3. Generate Classification Report
print("--- Classification Report ---")
print(classification_report(actuals, predictions))

# 4. Generate Confusion Matrix
cm = confusion_matrix(actuals, predictions)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(list(set(predictions))),
            yticklabels=sorted(list(set(actuals))))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: ASL Model Performance')
plt.show()

# Custom Model

# Building Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ASLModel(nn.Module):
    def __init__(self, num_classes):
        super(ASLModel, self).__init__()
        # 6-Layer Architecture: 4 Conv Layers, 2 Fully Connected Layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 512) # Assuming input images are 64x64
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))
        x = x.view(-1, 128 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
import shutil
import os

# Path to the offending folder
checkpoint_path = '/content/asl_video_ready'

if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
    print("✅ Removed .ipynb_checkpoints folder.")
else:
    print("ℹ️ .ipynb_checkpoints not found, proceeding.")

ℹ️ .ipynb_checkpoints not found, proceeding.


# Loading Data

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load datasets
train_dataset = datasets.ImageFolder(root='/content/asl_video_ready/train', transform=transform)
val_dataset = datasets.ImageFolder(root='/content/asl_video_ready/val', transform=transform)

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"✅ Loaded {len(train_dataset)} training videos and {len(val_dataset)} validation videos.")

✅ Loaded 47 training videos and 47 validation videos.


# Training Model From Scratch

In [ ]:
import torch
import torch.optim as optim

# Initialize lists to store metrics
train_losses = []
val_losses = []
val_accuracies = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Define Model: The neural network architecture

# Use the length of the dataset classes directly
num_classes = len(train_dataset.classes)
print(f"Initializing model for {num_classes} classes.")

# Initialize the model with the correct count
model = ASLModel(num_classes=num_classes).to(device)

# 2. Define Criterion: The loss function (CrossEntropy is standard for classification)
criterion = nn.CrossEntropyLoss()

# 3. Define Optimizer: The algorithm that updates model weights (Adam is recommended)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Assuming model, criterion, optimizer, train_loader, and val_loader are defined
print("🚀 Starting training with loss tracking...")

for epoch in range(20): # Set to your desired number of epochs
    # --- Training Phase ---
    model.train()
    running_train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    # Calculate average training loss for the epoch
    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # --- Validation Phase ---
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_acc = 100 * correct / total

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_acc)

    print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_acc:.2f}%")

# Save the final model
torch.save(model.state_dict(), '/content/final_asl_video_model.pth')
print("\n✅ Training complete and model saved.")

Initializing model for 47 classes.
🚀 Starting training with loss tracking...
Epoch 1 | Train Loss: 3.8732 | Val Loss: 3.8499 | Val Acc: 2.13%
Epoch 2 | Train Loss: 3.8519 | Val Loss: 3.8489 | Val Acc: 4.26%

✅ Training complete and model saved.


# Visualization

In [ ]:
import matplotlib.pyplot as plt

# 1. Define the function first
def plot_metrics(train_losses, val_losses, val_accuracies):
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(train_losses, label='Train Loss', color='blue')
    ax1.plot(val_losses, label='Val Loss', color='orange')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend(loc='upper left')

    ax2 = ax1.twinx()
    ax2.plot(val_accuracies, label='Val Accuracy', color='green', linestyle='--')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend(loc='upper right')

    plt.title('Training Performance Metrics')
    plt.show()

# 2. RUN TRAINING LOOP HERE...
# (The code I provided in the previous response populates
# train_losses, val_losses, and val_accuracies lists inside the loop)

# 3. CALL THE FUNCTION AFTER THE LOOP FINISHES
plot_metrics(train_losses, val_losses, val_accuracies)

# Testing

In [ ]:
import torch
import os
import shutil
from torchvision import transforms
from PIL import Image

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Define Model Architecture (Ensure this matches your training structure exactly)
# Make sure you have the 'ASLModel' class definition in this cell as well
model = ASLModel(num_classes=num_classes)
model.load_state_dict(torch.load('/content/final_asl_video_model.pth', map_location=device))
model.to(device)
model.eval()

# 3. Define Transforms (Must match training normalization)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 4. Map class indices to names
# This assumes your train folders are sorted A-Z
class_names = sorted(os.listdir('/content/asl_video_ready/train'))
idx_to_class = {i: name for i, name in enumerate(class_names)}

# 5. Testing logic
test_data_path = '/content/asl_video_ready/test'
results_folder = '/content/results_video_custom'

print(f"🚀 Starting inference on {test_data_path}...")

for class_label in os.listdir(test_data_path):
    class_path = os.path.join(test_data_path, class_label)

    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
                img_path = os.path.join(class_path, img_name)

                # Inference
                img = Image.open(img_path).convert('RGB')
                input_tensor = transform(img).unsqueeze(0).to(device)

                with torch.no_grad():
                    outputs = model(input_tensor)
                    _, predicted = torch.max(outputs, 1)
                    pred_label = idx_to_class[predicted.item()]

                # Sort into results folder
                target_folder = os.path.join(results_folder, f"pred_{pred_label}")
                os.makedirs(target_folder, exist_ok=True)
                shutil.copy2(img_path, os.path.join(target_folder, f"actual_{class_label}_{img_name}"))

print(f"\n🎉 Testing complete! Results sorted into: {results_folder}")

🚀 Starting inference on /content/asl_video_ready/test...

🎉 Testing complete! Results sorted into: /content/results_video_custom


# Visualization

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Update this path if your results are in a different location (e.g., '/content/results')
results_root = '/content/results_video_custom'

if not os.path.exists(results_root):
    print(f"Directory {results_root} not found. Please ensure your inference script successfully created this folder.")
else:
    actuals = []
    predictions = []

    # Parse folders: folder is 'pred_X', file is 'actual_Y_...'
    for pred_folder in os.listdir(results_root):
        if pred_folder.startswith('pred_'):
            pred_label = pred_folder.replace('pred_', '')
            folder_path = os.path.join(results_root, pred_folder)

            for img_name in os.listdir(folder_path):
                if img_name.startswith('actual_'):
                    parts = img_name.split('_')
                    if len(parts) >= 2:
                        actual_label = parts[1]
                        actuals.append(actual_label)
                        predictions.append(pred_label)

    if len(actuals) > 0:
        # Print Classification Report
        print("--- Classification Report ---")
        print(classification_report(actuals, predictions))

        # Generate Confusion Matrix
        labels = sorted(list(set(actuals) | set(predictions)))
        cm = confusion_matrix(actuals, predictions, labels=labels)

        # Plot Confusion Matrix
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=labels, yticklabels=labels)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title('Confusion Matrix: Custom ASL Model')
        plt.tight_layout()
        plt.savefig('confusion_matrix.png')
        print("✅ Visualization saved as confusion_matrix.png")
    else:
        print("No valid files found to generate metrics.")

# **Github Repo**

https://github.com/KVAlwaysLearning/Sign_Language_Detection_Sub

# **Streamlit App**

https://signlanguagedetectionsub-app.streamlit.app/